# Run the prediction

The approach is devided in two steps:

1) Cell division detection using semantic segmentation to detect the center of the division
2) Cell attribute regression on detected divisions

**What it does**  
- Lets you set the folders the script needs (segmentation model, regression model, folder with input data, folder to store the prediction), and verifies your paths exist and are readable.  
- Add the movie scale (in µm/pix) to a parameter file
- Runs the inference
- Saves the output into graphical representations and stores the results into variables for potential analysis

**Run each cell from top to bottom** (Shift+Enter).

---

In [1]:
from pathlib import Path
import json
import subprocess, sys
from shlex import quote
from dare3d.utils.utils import load_inference_results
import os

c:\Users\Qazi\anaconda3\envs\dare3d\lib\site-packages\torchmetrics\__init__.py:31: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.23.4)
  import scipy.signal


## 1) Prerequisites checklist

- You need to install the package in an environment, following https://github.com/JFRupprecht-OM/DARE3Dv2/tree/main

 
- Four folders are needed :
  - `segmentation.model_dir`: folder with the segmentation weights (position of each division)
  - `regression.model_dir`: folder with the regression weights (orientation of the detected division)
  - `inference_dir`: folder with the input data (one movie, tif file, [TZYX] format) for prediction.
  - `output_dir`: folder where the output data will be saved, which consists of i) the probability map of the division detection (tif file, same format than the input data), ii) a graphical representation of the division orientation  (tif file, same format than the input data) and iii) a npz file, which stores for each vector linking the two daughter cells: the center, the orientation (rotation matrix) and the distance between the two daughter cells (vector length)


## 2) Set your folders here
You can paste Windows paths with backslashes; we normalize them automatically.

In [2]:
segmentation_model_dir = r"C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\weights\segmentation3d_exp10-b"
regression_model_dir   = r"C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\weights\regression3d_exp10-b"
inference_dir          = r"C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\test_input"
output_dir            = r"C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\output"

predict_script         = r"dare3d\predict.py"


def _norm(p):  # normalize to absolute Path
    if p is None:
        return None
    return Path(p).expanduser().resolve()

seg_dir = _norm(segmentation_model_dir)
reg_dir = _norm(regression_model_dir)
inf_dir = _norm(inference_dir)
script  = _norm(predict_script)
out_dir = _norm(output_dir)

errors = []
if not script.exists():
    errors.append(f"Could not find predict script at: {script}")
if not seg_dir.exists():
    errors.append(f"Could not find segmentation.model_dir: {seg_dir}")
if not reg_dir.exists():
    errors.append(f"Could not find regression.model_dir: {reg_dir}")
if not inf_dir.exists():
    errors.append(f"Could not find inference_dir: {inf_dir}")
if not out_dir.exists():
    errors.append(f"Output directory does not exist: {out_dir}")

if errors:
    print("❌ Some paths are missing:\n- " + "\n- ".join(errors))
    raise SystemExit("Please fix the paths above and re-run this cell.")


print("✅ Paths are validated")
print(f"- predict.py     : {script}")
print(f"- segmentation   : {seg_dir}")
print(f"- regression     : {reg_dir}")
print(f"- inference data : {inf_dir}")
print(f"- output dir     : {out_dir}")

#make sure there is only one tif file in the inference dir
tif_files = list(Path(inference_dir).glob("*.tif"))
if len(tif_files) > 1:
    raise SystemExit(f"Please make sure there is exactly one .tif file in the inference_dir: {inference_dir}. Found {len(tif_files)} files.")
elif len(tif_files) == 0:
    raise SystemExit(f"The inference_dir: {inference_dir} contains 0 tif file")
movie_name = tif_files[0].name
movie_name = movie_name.replace(".tif","")
print("Movie name:", movie_name)

✅ Paths are validated
- predict.py     : C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare3d\predict.py
- segmentation   : C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\weights\segmentation3d_exp10-b
- regression     : C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\weights\regression3d_exp10-b
- inference data : C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\test_input
- output dir     : C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\output
Movie name: Image3_4-post


# 3) Input the scale (ZYX) of your movie 
The regression will run on isotropized data, with object ranging 10-20 pixels

In [3]:
scale_movie = (1,1,1)


scales_path = rf'scales\scales.json'
with open(scales_path, "r") as f:
    scales = json.load(f)
    if movie_name in scales and tuple(scales[movie_name]) != scale_movie:
        print(f"Previous scale for {movie_name}: {tuple(scales[movie_name])}, now updated to {scale_movie}")
    else :
        print(f"Setting scale for {movie_name} to {scale_movie}")
        scales[movie_name] = scale_movie

with open(scales_path, "w") as f:
    json.dump(scales, f, indent=4)

Setting scale for Image3_4-post to (1, 1, 1)


## 4) Run the prediction

This will start the process and stream its output below. 
You can safely stop it by choosing *Kernel → Interrupt* if needed.

In [4]:
cmd = [
    sys.executable,
    str(script),
    f"hydra.run.dir={out_dir}",
    f"segmentation.model_dir={seg_dir}",
    f"regression.model_dir={reg_dir}",
    f"inference_dir={inf_dir}"
]

print("Command preview:")
print(" ".join(quote(str(c)) for c in cmd))

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
)
for line in process.stdout:
    sys.stdout.write(line)

Command preview:
'c:\Users\Qazi\anaconda3\envs\dare3d\python.exe' 'C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare3d\predict.py' 'hydra.run.dir=C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\output' 'segmentation.model_dir=C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\weights\segmentation3d_exp10-b' 'regression.model_dir=C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\weights\regression3d_exp10-b' 'inference_dir=C:\Users\Qazi\Documents\PhD_Y1\Division3D\DARE3d\dare_data\test_input'
c:\Users\Qazi\anaconda3\envs\dare3d\lib\site-packages\torchmetrics\__init__.py:31: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 1.23.4)
  import scipy.signal
c:\Users\Qazi\anaconda3\envs\dare3d\lib\site-packages\monai\utils\module.py:396: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-3

# 5) Stores the results

In [5]:
# The probability map of each division center and a graphical representation of the division orientations are stored in the output directory.and
# The code below opens the npz file and stores the vectors centers, rotation matrices and lengths in variables for potential analysis

path_results = os.path.join(out_dir, "regression", "raw_predictions.npz")
results = load_inference_results(path_results)

centers = results["centers"][:, 1:]
rotation_matrices = results["rotation_matrices"]
lengths = results["lengths"]

print("Centers:", centers.shape) #T,X,Y,Z
print("Rotation Matrices:", rotation_matrices.shape)
print("Lengths:", lengths.shape)

Centers: (30, 4)
Rotation Matrices: (30, 3, 3)
Lengths: (30, 1)
